# Fase 2: Fundamentos de ML para Trading

Este notebook te guía paso a paso por los conceptos de Machine Learning
aplicados a datos de trading (OHLCV). Al completarlo tendrás:

1. Feature engineering con datos de velas
2. Un modelo de clasificación (LightGBM) entrenado
3. Evaluación con walk-forward (NO random split)
4. Entendimiento de por qué predecir precios es difícil

**Requisitos:** `pip install pandas numpy scikit-learn lightgbm matplotlib`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb
import matplotlib.pyplot as plt

print("Imports OK")

## 1. Cargar datos OHLCV

Si ya descargaste datos con Freqtrade (Fase 1), cárgalos directamente.
Si no, descargamos de una fuente pública.

In [ ]:
# Opción A: Cargar datos de Freqtrade (si ya los descargaste)
# df = pd.read_feather('../user_data/data/binance/BTC_USDT-1h.feather')

# Opción B: Generar datos sintéticos para practicar
# (Reemplaza con datos reales cuando los tengas)
np.random.seed(42)
n = 5000  # 5000 velas de 1h ≈ 208 días

# Simulación de precios con random walk
returns = np.random.normal(0.0001, 0.01, n)
close = 30000 * np.cumprod(1 + returns)

df = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=n, freq='1h'),
    'open': close * (1 + np.random.uniform(-0.005, 0.005, n)),
    'high': close * (1 + np.abs(np.random.normal(0, 0.008, n))),
    'low': close * (1 - np.abs(np.random.normal(0, 0.008, n))),
    'close': close,
    'volume': np.random.lognormal(10, 1, n)
})

print(f"Datos: {len(df)} velas")
print(f"Periodo: {df['date'].iloc[0]} a {df['date'].iloc[-1]}")
df.head()

## 2. Feature Engineering

Creamos indicadores técnicos como features para el modelo ML.

**Regla clave:** NUNCA uses el precio absoluto como feature. Usa retornos, ratios e indicadores normalizados.

In [ ]:
def add_features(df):
    """Agrega features técnicos al DataFrame."""
    # Retornos porcentuales
    df['return_1'] = df['close'].pct_change(1)
    df['return_4'] = df['close'].pct_change(4)
    df['return_12'] = df['close'].pct_change(12)
    df['return_24'] = df['close'].pct_change(24)
    
    # EMAs ratio (precio relativo a la media)
    for period in [10, 20, 50]:
        ema = df['close'].ewm(span=period).mean()
        df[f'ema_ratio_{period}'] = df['close'] / ema
    
    # RSI simplificado
    for period in [14, 28]:
        delta = df['close'].diff()
        gain = delta.clip(lower=0).rolling(period).mean()
        loss = (-delta.clip(upper=0)).rolling(period).mean()
        rs = gain / (loss + 1e-10)
        df[f'rsi_{period}'] = 100 - (100 / (1 + rs))
    
    # Volatilidad
    df['volatility_12'] = df['return_1'].rolling(12).std()
    df['volatility_24'] = df['return_1'].rolling(24).std()
    
    # Volumen relativo
    df['volume_ratio_20'] = df['volume'] / df['volume'].rolling(20).mean()
    
    # Rango de vela normalizado
    df['candle_range'] = (df['high'] - df['low']) / df['close']
    
    # Cuerpo de vela normalizado
    df['candle_body'] = (df['close'] - df['open']) / df['close']
    
    # Hora del día
    df['hour'] = df['date'].dt.hour
    
    return df

df = add_features(df)
print(f"Features creados: {len(df.columns)} columnas")
df.dropna(inplace=True)
print(f"Filas después de dropna: {len(df)}")

## 3. Definir el Target (lo que queremos predecir)

Target: ¿El precio sube más de 1% en las próximas 4 velas?
- 1 = Sí (comprar sería rentable)
- 0 = No

In [ ]:
# Target: retorno futuro en 4 velas > 1%
HORIZON = 4  # velas hacia el futuro
THRESHOLD = 0.01  # 1% de subida

df['future_return'] = df['close'].shift(-HORIZON) / df['close'] - 1
df['target'] = (df['future_return'] > THRESHOLD).astype(int)

# Eliminar filas sin target (las últimas HORIZON filas)
df = df.dropna(subset=['target'])

print(f"Distribución del target:")
print(df['target'].value_counts(normalize=True))
print(f"\nTotal filas: {len(df)}")

## 4. Walk-Forward Split (NO random split)

**CRÍTICO:** En series temporales NUNCA hacemos random split.
Siempre entrenamos con datos del PASADO y validamos con datos del FUTURO.

```
Fold 1: [TRAIN ████████] [TEST ██]
Fold 2: [TRAIN ██████████████] [TEST ██]
Fold 3: [TRAIN ████████████████████] [TEST ██]
```

In [ ]:
# Definir features y target
feature_cols = [
    'return_1', 'return_4', 'return_12', 'return_24',
    'ema_ratio_10', 'ema_ratio_20', 'ema_ratio_50',
    'rsi_14', 'rsi_28',
    'volatility_12', 'volatility_24',
    'volume_ratio_20',
    'candle_range', 'candle_body',
    'hour'
]

X = df[feature_cols].values
y = df['target'].values

# Walk-forward con TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Entrenar LightGBM
    model = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=20,
        verbose=-1
    )
    model.fit(X_train, y_train)
    
    # Predecir
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append(acc)
    
    print(f"Fold {fold+1}: Accuracy = {acc:.4f} | Train size = {len(train_idx)} | Test size = {len(test_idx)}")

print(f"\nAccuracy promedio: {np.mean(results):.4f} ± {np.std(results):.4f}")
print(f"\nNOTA: Si accuracy es ~50%, el modelo no puede predecir mejor que azar.")
print(f"Esto es NORMAL y esperado. Predecir precios es extremadamente difícil.")

## 5. Feature Importance

¿Qué features usa más el modelo? Si features irrelevantes dominan, hay ruido.

In [ ]:
# Entrenar modelo final con todos los datos (para inspección)
model_full = lgb.LGBMClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05, verbose=-1
)
model_full.fit(X, y)

# Feature importance
importance = pd.Series(
    model_full.feature_importances_,
    index=feature_cols
).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importance.plot(kind='barh')
plt.title('Feature Importance (LightGBM)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
print(importance.tail(5))

## 6. Simulación de Trading Simple

Convertimos las predicciones en un P&L simulado para ver si el modelo
genera valor real, no solo accuracy.

In [ ]:
# Usar el último fold como ejemplo de trading
train_idx, test_idx = list(tscv.split(X))[-1]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

model_sim = lgb.LGBMClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05, verbose=-1
)
model_sim.fit(X_train, y_train)

# Predicciones y retornos reales
predictions = model_sim.predict(X_test)
test_returns = df.iloc[test_idx]['future_return'].values

# P&L: solo entramos cuando el modelo predice 1 (subida)
COMMISSION = 0.001  # 0.1% por trade (entry + exit = 0.2% total)

pnl = []
for pred, ret in zip(predictions, test_returns):
    if pred == 1:  # El modelo dice: comprar
        trade_pnl = ret - (2 * COMMISSION)  # Retorno menos comisiones ida y vuelta
        pnl.append(trade_pnl)
    else:
        pnl.append(0)  # No hacemos nada

cumulative_pnl = np.cumsum(pnl)

# Gráfico
plt.figure(figsize=(12, 5))
plt.plot(cumulative_pnl * 100, label='P&L del modelo (%)')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.5)
plt.title('P&L Simulado del Modelo ML')
plt.xlabel('Trade #')
plt.ylabel('P&L acumulado (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

total_trades = sum(predictions)
total_pnl = sum(pnl) * 100
print(f"Total trades: {total_trades}")
print(f"P&L total: {total_pnl:.2f}%")
print(f"P&L por trade promedio: {total_pnl/max(total_trades,1):.4f}%")
print(f"\nSi el P&L es cercano a 0% o negativo, el modelo NO agrega valor.")
print(f"Esto es normal en un primer intento. Itera features y targets.")

## 7. Conclusiones y Próximos pasos

### Lo que deberías haber aprendido:
- Feature engineering con datos OHLCV
- Walk-forward validation (NUNCA random split en series temporales)
- LightGBM para clasificación
- Feature importance
- Que predecir precios es MUY difícil y ~50% accuracy es normal

### Próximos pasos (Fase 3):
1. Integrar este conocimiento en FreqAI (ver `strategies/FreqaiLightgbm.py`)
2. Experimentar con diferentes targets (retorno en 8 velas, umbral 0.5%, etc.)
3. Agregar más features: Bollinger Bands, ATR, datos de volumen avanzados
4. Probar CatBoost como alternativa a LightGBM